# Results, Discussion and Conclusion

## Research question and evaluation design

This analysis examines whether recent Tuen Mun Air Quality Health Index (AQHI) observations can identify whether the AQHI will be **at least 4 six hours later**. The positive class is therefore defined as target AQHI >= 4.

The comparison evaluates five classifiers—Decision Tree, Random Forest, Logistic Regression, K-Nearest Neighbours (KNN), and Support Vector Machine (SVM)—against a class-persistence baseline. The baseline carries the current AQHI classification forward as the six-hour prediction. The models use preceding AQHI history windows of 3, 6, or 12 hours.

All three history windows use the same matched complete-case cohort of 2,134 forecasts, including 581 (27.2%) positive targets. Forecast input times run from 1 April 2026 00:00 to 30 June 2026 18:00 HKT; corresponding target times run from 1 April 2026 06:00 to 1 July 2026 00:00 HKT. The training period runs from April 2025 through March 2026. Matching cases across windows supports a fair comparison of their test metrics.

The holdout period was inspected during model development. Accordingly, the test-set comparisons should be interpreted as exploratory evidence rather than as a fully independent final evaluation.

## Data and code workflow

The source files are in `outputs/aqhi4-lookback-comparison/`. The first code cell reads the exported metrics and the three row-level prediction files. It then:

1. Verifies that each recorded actual class agrees with the rule AQHI >= 4, and checks that all models and history windows use the same target timestamps and labels.
2. Recalculates TP, FN, FP, TN, precision, recall, F1, and accuracy from row-level predictions.
3. Compares the recalculated values and sample counts with `test_metrics.csv`.
4. Prints five selected six-hour prediction cases for closer interpretation.

Run the first code cell before the case-extraction cell, or use Run All. In VS Code, open the notebook from the project root so the relative file paths resolve.


In [1]:
from pathlib import Path
import csv

BASE = Path("outputs/aqhi4-lookback-comparison")
REQUIRED_FILES = [
    "test_metrics.csv",
    "test_predictions_3h.csv",
    "test_predictions_6h.csv",
    "test_predictions_12h.csv",
]
missing = [name for name in REQUIRED_FILES if not (BASE / name).is_file()]
if missing:
    raise FileNotFoundError(
        "Open/run this notebook from the project root. Missing files: "
        + ", ".join(missing)
    )

METHODS = [
    "Decision Tree", "Random Forest", "Logistic Regression",
    "KNN", "SVM", "Persistence baseline"
]
LOOKBACKS = ["3h", "6h", "12h"]


def read_csv(path):
    with path.open(encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


metrics_rows = read_csv(BASE / "test_metrics.csv")
prediction_rows = {
    lookback: read_csv(BASE / f"test_predictions_{lookback}.csv")
    for lookback in LOOKBACKS
}
metrics_by_key = {(r["lookback"], r["method"]): r for r in metrics_rows}
if len(metrics_by_key) != len(LOOKBACKS) * len(METHODS):
    raise ValueError("Expected exactly one test_metrics.csv row per method and window.")

# Check the shared cohort and threshold labels, then recompute all reported metrics.
cohort_by_window = {}
recomputed = {}
label_checks = []
for lookback, records in prediction_rows.items():
    cohort_by_method = {}
    for method in METHODS:
        selected = [r for r in records if r["method"] == method]
        if len(selected) != 2134:
            raise ValueError(f"Unexpected sample count for {lookback}, {method}: {len(selected)}")
        labels = {r["target_time"]: r["actual_class"] for r in selected}
        if len(labels) != len(selected):
            raise ValueError(f"Duplicate target timestamps for {lookback}, {method}.")
        cohort_by_method[method] = labels
        label_checks.extend(
            int(float(r["actual_aqhi"]) >= 4) == int(r["actual_class"])
            for r in selected
        )

        tp = sum(r["actual_class"] == "1" and r["predicted_class"] == "1" for r in selected)
        fn = sum(r["actual_class"] == "1" and r["predicted_class"] == "0" for r in selected)
        fp = sum(r["actual_class"] == "0" and r["predicted_class"] == "1" for r in selected)
        tn = sum(r["actual_class"] == "0" and r["predicted_class"] == "0" for r in selected)
        precision = tp / (tp + fp) if tp + fp else 0
        recall = tp / (tp + fn) if tp + fn else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
        accuracy = (tp + tn) / len(selected)
        recomputed[(lookback, method)] = {
            "samples": len(selected), "positive_samples": tp + fn,
            "tp": tp, "fn": fn, "fp": fp, "tn": tn,
            "precision": precision, "recall": recall,
            "f1": f1, "accuracy": accuracy,
        }
    reference = cohort_by_method[METHODS[0]]
    if any(labels != reference for labels in cohort_by_method.values()):
        raise ValueError(f"Methods do not share the same cases in {lookback}.")
    cohort_by_window[lookback] = reference

if any(cohort != cohort_by_window[LOOKBACKS[0]] for cohort in cohort_by_window.values()):
    raise ValueError("The three history windows do not share the same target cohort.")
if not all(label_checks):
    raise ValueError("At least one actual_class label disagrees with the AQHI >= 4 rule.")

checks = []
for row in metrics_rows:
    calc = recomputed[(row["lookback"], row["method"])]
    count_fields = {
        "samples": "samples", "positive_samples": "positive_samples",
        "true_positive": "tp", "false_negative": "fn",
        "false_positive": "fp", "true_negative": "tn",
    }
    counts_match = all(int(row[field]) == calc[key] for field, key in count_fields.items())
    rates_match = all(
        abs(calc[field] - float(row[field])) < 1e-10
        for field in ("precision", "recall", "f1", "accuracy")
    )
    checks.append(counts_match and rates_match)

if not all(checks):
    raise ValueError("Recalculated predictions do not match test_metrics.csv.")

print(
    f"Metric rows checked: {len(metrics_rows)}; all counts and rates match: {all(checks)}"
)
print(
    f"Threshold labels match AQHI >= 4: {all(label_checks)}; "
    f"same target cohort across windows: {len(cohort_by_window[LOOKBACKS[0]])}"
)
for lookback in LOOKBACKS:
    positive_count = sum(value == "1" for value in cohort_by_window[lookback].values())
    print(f"{lookback}: 2,134 forecasts; {positive_count} positive targets")


Metric rows checked: 18; all counts and rates match: True
Threshold labels match AQHI >= 4: True; same target cohort across windows: 2134
3h: 2,134 forecasts; 581 positive targets
6h: 2,134 forecasts; 581 positive targets
12h: 2,134 forecasts; 581 positive targets


## Results: comparison of methods and history windows

Precision is TP / (TP + FP), the proportion of positive predictions that are correct. Recall is TP / (TP + FN), the proportion of AQHI >= 4 target hours detected. F1 is the harmonic mean of precision and recall. In the table, TP (true positive) is a correctly detected AQHI >= 4 target; FN (false negative) is a missed AQHI >= 4 target; FP (false positive) is an incorrect positive alert; and TN (true negative) is a correctly identified target below AQHI 4.

The table reports test-set metrics for all methods and lookback windows. Values are rounded to three decimal places.

| History | Method | Precision | Recall | F1 | TP | FN | FP |
|---|---|---:|---:|---:|---:|---:|---:|
| 3h | Decision Tree | 0.711 | 0.702 | 0.706 | 408 | 173 | 166 |
| 3h | Random Forest | 0.666 | 0.793 | 0.724 | 461 | 120 | 231 |
| 3h | Logistic Regression | 0.703 | 0.737 | 0.719 | 428 | 153 | 181 |
| 3h | KNN | 0.649 | 0.849 | 0.735 | 493 | 88 | 267 |
| 3h | SVM | 0.682 | 0.754 | 0.716 | 438 | 143 | 204 |
| 3h | Persistence baseline | 0.692 | 0.692 | 0.692 | 402 | 179 | 179 |
| 6h | Decision Tree | 0.578 | 0.921 | 0.710 | 535 | 46 | 390 |
| 6h | Random Forest | 0.681 | 0.812 | **0.741** | 472 | 109 | 221 |
| 6h | Logistic Regression | **0.711** | 0.728 | 0.719 | 423 | 158 | 172 |
| 6h | KNN | 0.642 | **0.840** | 0.728 | 488 | 93 | 272 |
| 6h | SVM | 0.680 | 0.771 | 0.723 | 448 | 133 | 211 |
| 6h | Persistence baseline | 0.692 | 0.692 | 0.692 | 402 | 179 | 179 |
| 12h | Decision Tree | **0.752** | 0.642 | 0.693 | 373 | 208 | 123 |
| 12h | Random Forest | 0.746 | 0.678 | 0.711 | 394 | 187 | 134 |
| 12h | Logistic Regression | 0.715 | 0.695 | 0.705 | 404 | 177 | 161 |
| 12h | KNN | 0.625 | 0.836 | 0.716 | 486 | 95 | 291 |
| 12h | SVM | 0.681 | 0.771 | 0.723 | 448 | 133 | 210 |
| 12h | Persistence baseline | 0.692 | 0.692 | 0.692 | 402 | 179 | 179 |

![Test F1 by history window](outputs/aqhi4-lookback-comparison/lookback_f1_comparison.png)

### Precision-recall trade-off at six hours

![Precision-recall plot for six-hour forecasts](outputs/results-discussion/precision_recall_6h.png)

Each point represents one method; the axes show precision and recall, and the label gives its F1. Random Forest has the strongest F1 balance. Decision Tree reaches higher recall but substantially lower precision.

### False positives and false negatives

![Six-hour false-positive and false-negative counts](outputs/results-discussion/error_counts_6h.png)

This plot makes the error trade-off explicit: Decision Tree misses fewer high-AQHI hours, but produces more false alerts.

### Six-hour results

Among the six-hour models, Random Forest achieved the highest F1 (0.741), an absolute increase of 0.049 over the persistence baseline (0.692). Its precision was 0.681 and recall was 0.812, with 472 true positives, 109 false negatives, and 221 false positives.

Decision Tree achieved the highest recall (0.921), detecting 535 of 581 positive target hours. However, it also produced 390 false positives and had the lowest six-hour precision (0.578). Logistic Regression had the highest six-hour precision (0.711), while KNN had the second-highest recall (0.840). Thus, model ranking depends on the relative cost assigned to missed high-AQHI hours and false alerts.

### Effect of lookback window

The effect of history length varies by method. Random Forest F1 increased from 0.724 with 3h history to 0.741 with 6h history. Logistic Regression was nearly unchanged between 3h and 6h (0.719 for both), while KNN performed best with 3h history (0.735) and declined to 0.728 at 6h and 0.716 at 12h. Decision Tree performed best at 6h (0.710). SVM performed similarly across windows, with F1 ranging from 0.716 to 0.723; its 12h result was only marginally above its six-hour result.

Overall, no single history window improved every classifier. The persistence baseline was constant across windows, as expected.


## Error analysis: selected six-hour forecasts

The examples below show selected six-hour predictions. Forecast issued is the input time; target time is six hours later. A prediction of 1 indicates that the method forecasts target AQHI >= 4.

| Forecast issued (HKT) | Target time (HKT) | Actual AQHI | Decision Tree | Random Forest | Logistic Regression | KNN | SVM | Persistence |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| 25 Apr 09:00 | 25 Apr 15:00 | 7 | 1 | 0 | 0 | 0 | 0 | 0 |
| 25 Apr 10:00 | 25 Apr 16:00 | 8 | 1 | 1 | 1 | 1 | 0 | 0 |
| 25 Apr 11:00 | 25 Apr 17:00 | 8 | 1 | 1 | 0 | 1 | 0 | 0 |
| 29 May 09:00 | 29 May 15:00 | 7 | 0 | 0 | 0 | 0 | 0 | 0 |
| 19 May 08:00 | 19 May 14:00 | 2 | 1 | 1 | 1 | 1 | 1 | 0 |

![Classification outcomes for selected six-hour forecast cases](outputs/results-discussion/selected_cases_6h.png)

In the heatmap, TP (true positive) marks a correctly detected AQHI >= 4 target; FN (false negative) marks a missed AQHI >= 4 target; FP (false positive) marks a false alert for a target below 4; and TN (true negative) marks a correctly identified target below 4.

On 25 April, the target AQHI was 8 at 16:00; Decision Tree, Random Forest, Logistic Regression, and KNN issued positive predictions, whereas SVM and the persistence baseline did not. At 15:00, when the target AQHI was 7, only Decision Tree issued a positive prediction. On 29 May, all six methods failed to flag the target AQHI of 7 at 15:00. On 19 May, all five classifiers predicted AQHI >= 4 for a target AQHI of 2, while the persistence baseline did not. These examples illustrate that aggregate metrics can conceal important differences in performance on individual hours.

Adjacent hourly targets may belong to the same pollution episode. Therefore, these rows should be interpreted as timestamp-level examples rather than as independent events.


In [2]:
# Rebuild selected six-hour examples directly from the prediction file.
case_targets = {
    "2026-04-25 15:00:00+08:00",
    "2026-04-25 16:00:00+08:00",
    "2026-04-25 17:00:00+08:00",
    "2026-05-29 15:00:00+08:00",
    "2026-05-19 14:00:00+08:00",
}
cases = {}
for r in prediction_rows["6h"]:
    if r["target_time"] in case_targets:
        item = cases.setdefault(r["target_time"], {
            "forecast_issued": r["timestamp"],
            "actual_aqhi": r["actual_aqhi"],
            "predictions": {}
        })
        item["predictions"][r["method"]] = r["predicted_class"]

if set(cases) != case_targets or any(
    set(case["predictions"]) != set(METHODS) for case in cases.values()
):
    raise ValueError("Selected case timestamps or method predictions are incomplete.")

for target in sorted(cases):
    item = cases[target]
    print(target, "AQHI", item["actual_aqhi"],
          " ".join(f"{m}={item['predictions'][m]}" for m in METHODS))


2026-04-25 15:00:00+08:00 AQHI 7.0 Decision Tree=1 Random Forest=0 Logistic Regression=0 KNN=0 SVM=0 Persistence baseline=0
2026-04-25 16:00:00+08:00 AQHI 8.0 Decision Tree=1 Random Forest=1 Logistic Regression=1 KNN=1 SVM=0 Persistence baseline=0
2026-04-25 17:00:00+08:00 AQHI 8.0 Decision Tree=1 Random Forest=1 Logistic Regression=0 KNN=1 SVM=0 Persistence baseline=0
2026-05-19 14:00:00+08:00 AQHI 2.0 Decision Tree=1 Random Forest=1 Logistic Regression=1 KNN=1 SVM=1 Persistence baseline=0
2026-05-29 15:00:00+08:00 AQHI 7.0 Decision Tree=0 Random Forest=0 Logistic Regression=0 KNN=0 SVM=0 Persistence baseline=0


## Discussion

The six-hour Random Forest provides the strongest balance of precision and recall among the evaluated six-hour methods, as measured by F1. It detects 472 of 581 positive target hours and produces 221 false alerts. The six-hour Decision Tree may be preferable when reducing missed positive hours is the primary objective: it misses 46 positive hours, compared with 109 for Random Forest, but produces 390 false alerts. The appropriate choice therefore depends on the intended use and the relative consequences of missed alerts and false alarms.

The lookback comparison does not support a general claim that more historical data improves performance. Random Forest benefits from extending history from 3h to 6h, KNN performs best with 3h, and SVM varies little across the three windows. Because parameters and classification thresholds were selected separately for each method and window, the comparison evaluates tuned configurations rather than isolating the effect of any single additional lag feature.

## Limitations

- **Geographic and temporal coverage:** The evaluation uses one monitoring station and a three-month test period (April–June 2026). Performance may differ at other locations or seasons.
- **Hourly dependence:** Consecutive target hours may belong to the same pollution episode and are autocorrelated. The reported hourly counts should not be interpreted as counts of independent pollution events.
- **Input coverage:** Models use AQHI history and time-of-day features; meteorological, emissions, and other potentially informative variables are not included.
- **Missing and substituted readings:** The comparison follows the repository's complete-case cohort and flagged-reading policy. Missing hours were not interpolated. Results should be interpreted with this data policy in mind.
- **Exploratory holdout:** The test period had been inspected during model development. These results may be optimistic and should be confirmed on a later, untouched period.
- **Operational evaluation:** The analysis reports hour-level classification metrics. A future evaluation should also consider episode-level detection, warning lead time, and the burden of repeated false alerts.

## Conclusion

1. **Best overall six-hour balance:** Random Forest achieved the highest six-hour F1 (0.741; precision 0.681; recall 0.812), exceeding the persistence baseline F1 (0.692) by 0.049.
2. **Highest detection rate:** Decision Tree achieved the highest recall (0.921), but its 390 false positives indicate a substantial false-alert burden.
3. **History-window effect:** There is no universally best lookback window. Six-hour history gave Random Forest its best F1, while KNN performed best with three-hour history; SVM changed little across windows.
4. **Scope of inference:** These findings apply to the matched Tuen Mun test cohort and are exploratory because the holdout period had already been inspected.

On this test cohort, six-hour Random Forest is the most balanced candidate for identifying AQHI >= 4 six hours ahead. This is a provisional finding rather than a deployment recommendation. Evaluation on a later, untouched period, with event-level measures and broader predictive inputs, is required before drawing operational conclusions.
